In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import col
import pyspark.sql.types as T

In [0]:
%sql
select 
    *
from olist_dataset.silver.delivered_orders

In [0]:
df = spark.read.table('olist_dataset.silver.delivered_orders')

display(df.limit(10))

In [0]:
df_with_days = df.withColumn('purchase_date',col('purchase_time').cast(T.DateType()))\
        .withColumn('customer_delivery_date', col('customer_delivery_date').cast(T.DateType()))\
        .withColumn('delivered_carrier_date',F.datediff(col('delivered_carrier_date'),col('purchase_time')))\
        .withColumn('customer_delivery_est_date',F.datediff(col('customer_delivery_est_date'),col('purchase_time')))\
        .withColumn('customer_delivery_days',F.datediff(col('customer_delivery_date'),col('purchase_time')))

In [0]:
df_with_renamed_cols = df_with_days.withColumnsRenamed({
    'purchase_time' : 'purchase_timestamp',
    'delivered_carrier_date' : 'days_to_reach_delivered_carrier',
    'customer_delivery_est_date' : 'est_days_to_delivery',
    'customer_delivery_date' : 'delivery_date'
})

In [0]:
df_with_renamed_cols.write.format('delta')\
    .mode('overwrite')\
    .option('mergeSchemea',True)\
    .saveAsTable('')